# LLM news sentiment and the Shanghai Composite equity risk premium (v2)

**Before you start**
1. Add three Colab Secrets (key icon on the left): `GEMINI_API_KEY`, `NYT_KEY`, `GUARDIAN_KEY`, with notebook access on.
2. Have `china_sentiment_LSEG_input_v2.xlsx` and your old `scores_cache.csv` ready to upload.
3. Runtime > Run all. You will be asked to upload twice: the input workbook (cell 4), then the score cache (cell 9).

Cells 17 (debate), 22 and 23 (bull, bear and judge) and 24 (kappa) are switched off by the first line `%%script false --no-raise-error`. Delete that line to run one.

In [ ]:
!pip -q install google-genai yfinance akshare statsmodels openpyxl scikit-learn

Pipeline overview

```
# =============================================================================
# LLM news sentiment and the Shanghai Composite equity risk premium
#
# Written for Google Colab. Each "# ===== CELL n =====" marker is one Colab cell.
# Running this file top to bottom as a script also works, except for the
# files.upload() calls, which are Colab only.
#
# Pipeline:
#   cells 1-6    setup, API keys, market data (Shanghai Composite, CN 1Y yield)
#   cells 7-10   headline collection (New York Times Archive, Guardian Open Platform)
#   cells 11-14  zero-shot scoring with the Ma, Lyu and Li (2024) prompt
#   cells 15-20  weekly aggregation, regressions, Newey-West, out-of-sample test
#   cell  21     Excel workbook output
#   cells 22-23  bull / bear / judge multi-agent scoring and comparison (optional, heavy on quota)
#   cell  24     blind two-rater hand check: agreement and Cohen's kappa
#
# Changes in this version: sample from Feb 2024, Google News market headlines on,
# NBS manufacturing PMI baseline (sheet pmi_monthly), Google-only robustness row,
# blind 50-headline label check with two raters. Skip cell 17 (debate).
# =============================================================================

# Colab: !pip -q install google-genai yfinance akshare statsmodels openpyxl
```

## Settings

In [ ]:
import os, re, time, glob, requests
import numpy as np
import pandas as pd

START = "2024-02-02"          # first week in the dataset (a Friday; LSEG data starts Jan 2024)
END   = "2026-11-20"          # last week (a Friday)
INDEX = "SSEC"                # "SSEC" (Shanghai Composite) or "CSI300"

# Headline sources. NYT and Guardian carry exact timestamps. Google News carries the date only.
USE_GOOGLE = True             # NYT + Guardian + Google News market headlines (main run)
TAG = "api_google" if USE_GOOGLE else "api_only"

MODEL = "gemini-3.5-flash-lite"   # check the exact name in AI Studio
BATCH_SIZE = 50               # headlines per request (free tier: ~500 requests per day)
REQUESTS_PER_MINUTE = 10      # stays under the free-tier per-minute limit
MAX_HEADLINES_PER_WEEK = 50   # random sample per week, fixed seed. None = use all.
TZ = "Asia/Shanghai"

# Outlets that get priority when headlines are capped per week (matched on the source name)
PREFERRED_SOURCES = ["new york times", "guardian", "reuters", "bloomberg", "cnbc", "south china morning post", "scmp",
                     "caixin", "financial times", "wall street journal", "wsj", "marketwatch", "nikkei",
                     "yahoo finance", "barron", "associated press", "economist", "fortune"]

# Google News (only used if USE_GOOGLE = True). One search per query per week.
GOOGLE_QUERIES = ['"China stocks" OR "Shanghai Composite" OR "CSI 300" OR "Chinese shares" OR "A-shares"',
                  'PBOC OR yuan OR "China stimulus" OR "China economy"']
YAHOO_INDEX = {"SSEC": "000001.SS", "CSI300": "000300.SS"}   # free fallback if the LSEG sheet is empty

# Ma, Lyu and Li (2024) prompt, unchanged from the assignment brief
PROMPT = ("Clear all prior contexts and instructions. As a finance expert, "
          "evaluate the potential effect of this news headline on the Chinese "
          "stock market. Using only the headline's information, predict whether "
          "the impact will be an increase (UP), a decrease (DOWN), or uncertain "
          "(UNCERTAIN). Choose only one option, UP, DOWN, or UNCERTAIN, and "
          "refrain from providing further comments.\n\nHeadline: {headline}")
SCORE_MAP = {"UP": 1, "DOWN": -1, "UNCERTAIN": 0}

WORK = "/content/china_sentiment" if os.path.exists("/content") else os.path.abspath("china_sentiment_run")
for d_ in ["raw/news_google", "output"]:
    os.makedirs(os.path.join(WORK, d_), exist_ok=True)
os.chdir(WORK)
CACHE = "scores_cache.csv"
OUTPUT_XLSX = f"output/china_sentiment_dataset_{TAG}.xlsx"
print("Working folder:", WORK, "| run tag:", TAG)

## API keys

In [ ]:
def get_key(name):
    k = None
    try:
        from google.colab import userdata
        k = userdata.get(name)
    except Exception:
        pass
    if not k:
        from getpass import getpass
        k = getpass(f"Paste {name}: ")
    return k.strip()

GEMINI_KEY = get_key("GEMINI_API_KEY")
NYT_KEY = get_key("NYT_KEY")
GUARDIAN_KEY = get_key("GUARDIAN_KEY")
os.environ["GEMINI_API_KEY"] = GEMINI_KEY
print("Keys set")

## Upload the LSEG input workbook

In [ ]:
INPUT = "input.xlsx"
from google.colab import files
up = files.upload()
with open(INPUT, "wb") as f:
    f.write(up[list(up)[0]])
print("Uploaded", list(up)[0])

sheets = pd.read_excel(INPUT, sheet_name=None)
print({k: len(v) for k, v in sheets.items()})

## Market data and PMI

In [ ]:
def fetch_start():
    return (pd.Timestamp(START) - pd.Timedelta(days=30)).strftime("%Y-%m-%d")

def sheet_series(name):
    """Finds a date column and the first numeric column after it, wherever LSEG put them."""
    if name not in sheets:
        return None
    raw = pd.read_excel(INPUT, sheet_name=name, header=None)
    if raw.dropna(how="all").empty:
        return None
    for i in range(raw.shape[1] - 1):
        col = raw.iloc[:, i]
        num = pd.to_numeric(col, errors="coerce")
        if num.notna().sum() >= 20 and num.dropna().between(20000, 80000).all() and (num.dropna() % 1 == 0).all():
            dt = pd.to_datetime(num, unit="D", origin="1899-12-30")      # Excel serial dates
        elif pd.api.types.is_numeric_dtype(col):
            continue
        else:
            dt = pd.to_datetime(col, errors="coerce", format="mixed")
        if dt.notna().sum() < 20:
            continue
        if getattr(dt.dt, "tz", None) is not None:
            dt = dt.dt.tz_localize(None)
        for j in range(i + 1, raw.shape[1]):
            v = pd.to_numeric(raw.iloc[:, j].astype(str).str.replace(",", "").str.rstrip("%"), errors="coerce")
            ok = dt.notna() & v.notna()
            if ok.sum() >= 20:
                s = pd.Series(v[ok].values, index=dt[ok].dt.normalize())
                return s[~s.index.duplicated(keep="last")].sort_index()
    print(f"WARNING: no date and value columns found in sheet {name}")
    return None

def free_index():
    import yfinance as yf
    df = yf.download(YAHOO_INDEX[INDEX], start=fetch_start(),
                     end=(pd.Timestamp(END) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
                     interval="1d", auto_adjust=False, progress=False)
    close = df["Close"]
    close = close.iloc[:, 0] if isinstance(close, pd.DataFrame) else close
    close.index = pd.to_datetime(close.index).tz_localize(None).normalize()
    return close.dropna()

def free_rf():
    import akshare as ak
    df = ak.rate_interbank(market="上海银行同业拆借市场", symbol="Shibor人民币", indicator="3月")
    col = "利率" if "利率" in df.columns else df.columns[1]
    return pd.Series(pd.to_numeric(df[col], errors="coerce").values,
                     index=pd.to_datetime(df[df.columns[0]])).sort_index().dropna()

px = sheet_series("index_daily")
if px is None:
    print("index_daily empty: downloading from Yahoo Finance"); px = free_index()
rf = sheet_series("rf_daily")
RF_LABEL = "China 1-year government yield (LSEG CN1YT=RR)"
if rf is None:
    print("rf_daily empty: downloading 3M SHIBOR via akshare"); rf = free_rf(); RF_LABEL = "3-month SHIBOR (akshare)"
if rf.median() < 0.2:                       # entered as decimals (0.0145)
    print("Rates look like decimals: converting to percent"); rf = rf * 100

pmi = None
if "pmi_monthly" in sheets:
    p_ = pd.read_excel(INPUT, sheet_name="pmi_monthly")
    pmi = pd.Series(pd.to_numeric(p_.iloc[:, 1], errors="coerce").values,
                    index=pd.to_datetime(p_.iloc[:, 0], errors="coerce")).dropna().sort_index()
    pmi = pmi[pmi.index.notna()]
    if pmi.empty:
        print("WARNING: pmi_monthly sheet has no values, PMI models will be skipped"); pmi = None
    else:
        print(f"PMI: {len(pmi)} months, {pmi.index.min().date()} to {pmi.index.max().date()}")
else:
    print("WARNING: no pmi_monthly sheet, PMI models will be skipped")

for name, s in [("Index", px), ("Rf", rf)]:
    print(f"{name}: {len(s)} days, {s.index.min().date()} to {s.index.max().date()}, last value {s.iloc[-1]:.3f}")
if px.index.min() > pd.Timestamp(START):
    print("WARNING: index data starts after START. Add earlier days or move START.")

## Weekly equity risk premium

In [ ]:
def weekly_market():
    p = px.sort_index()
    last_day = p.groupby(p.index.to_period("W-FRI")).apply(lambda s: s.index.max())
    wk = pd.DataFrame({"last_trading_day": last_day.values, "close": p.loc[last_day.values].values})
    wk["close_time"] = (pd.to_datetime(wk["last_trading_day"]) + pd.Timedelta(hours=15)).dt.tz_localize(TZ)
    r = rf.sort_index()
    r_daily = r.reindex(pd.date_range(min(r.index.min(), p.index.min()), p.index.max())).ffill()
    wk["rf_pct_start"] = r_daily.reindex(wk["last_trading_day"]).shift(1).values
    wk["ret_log"] = np.log(wk["close"] / wk["close"].shift(1))
    wk["rf_weekly"] = wk["rf_pct_start"] / 100 / 52
    wk["erp"] = wk["ret_log"] - wk["rf_weekly"]
    return wk

def week_index(ts, wk):
    """Index of the week each timestamp belongs to: first weekly close at or after the timestamp."""
    closes_utc = wk["close_time"].dt.tz_convert("UTC").dt.tz_localize(None).values
    t = pd.to_datetime(ts, utc=True).dt.tz_localize(None).values
    return np.searchsorted(closes_utc, t, side="left")

wk = weekly_market()
print(len(wk), "weekly closes,", wk["last_trading_day"].min().date(), "to", wk["last_trading_day"].max().date())

## Headline download functions

In [ ]:
from email.utils import parsedate_to_datetime
import xml.etree.ElementTree as ET
from urllib.parse import quote_plus

PREFIX = re.compile(r"^(UPDATE \d+[-:]\s*|REFILE[-:]\s*|CORRECTED[-:]\s*|WRAPUP \d+[-:]\s*|"
                    r"EXCLUSIVE[-:]\s*|ANALYSIS[-:]\s*|FOCUS[-:]\s*|BREAKINGVIEWS[-:]\s*)", re.I)
CN = re.compile(r"china|chinese|beijing|shanghai|yuan|pboc|hong kong|csi 300|xi jinping|shenzhen|a-share|hang seng", re.I)
EXCL = re.compile(r"south china sea", re.I)
COLS = ["timestamp_utc", "headline", "source"]

def headline_tz():
    rd = sheets.get("README")
    if rd is not None:
        for _, row in rd.iterrows():
            if str(row.iloc[0]).strip().lower() == "headline time zone":
                return str(row.iloc[1]).strip()
    return "UTC"

def clean_headlines(df):
    df = df.dropna(subset=["headline"]).copy()
    df["headline"] = df["headline"].astype(str).str.strip().str.replace(PREFIX, "", regex=True)
    df = df[df["headline"].str.len() > 15]
    df = df[~df["headline"].str.upper().str.startswith("BRIEF-")]
    df["key"] = df["headline"].str.lower().str.replace(r"[^a-z0-9 ]", "", regex=True).str.strip()
    df = df.sort_values("timestamp_utc").drop_duplicates("key", keep="first")
    return df.drop(columns="key").reset_index(drop=True)

def nyt_archive(key):
    """New York Times Archive API: one request per month, exact timestamps. 5 requests a minute, 500 a day."""
    f = "raw/news_nyt.csv"
    if os.path.exists(f):
        return pd.read_csv(f)
    rows = []
    for m in pd.period_range(pd.Timestamp(START) - pd.Timedelta(days=14), END, freq="M"):
        r = None
        for attempt in range(3):
            r = requests.get(f"https://api.nytimes.com/svc/archive/v1/{m.year}/{m.month}.json",
                             params={"api-key": key}, timeout=120)
            if r.status_code == 429:
                time.sleep(60); continue
            break
        if r is None or r.status_code != 200:
            print("NYT", m, getattr(r, "status_code", None), (r.text[:100] if r is not None else "")); time.sleep(12); continue
        for d_ in r.json()["response"]["docs"]:
            t = (d_.get("headline") or {}).get("main") or ""
            if CN.search(t):
                rows.append({"timestamp_utc": d_["pub_date"], "headline": t, "source": "New York Times"})
        print("NYT", m, len(rows)); time.sleep(12)
    df = pd.DataFrame(rows, columns=COLS); df.to_csv(f, index=False); return df

def guardian(key):
    """Guardian Open Platform: exact timestamps."""
    f = "raw/news_guardian.csv"
    if os.path.exists(f):
        return pd.read_csv(f)
    rows, page = [], 1
    while True:
        r = requests.get("https://content.guardianapis.com/search", params={
            "q": "China AND (economy OR stocks OR PBOC OR yuan OR tariffs OR stimulus OR markets)",
            "section": "business|world",
            "from-date": (pd.Timestamp(START) - pd.Timedelta(days=14)).strftime("%Y-%m-%d"),
            "to-date": END, "page-size": 200, "page": page,
            "order-by": "oldest", "api-key": key}, timeout=60)
        if r.status_code != 200:
            print("Guardian", r.status_code, r.text[:150]); break
        resp = r.json()["response"]
        for x in resp["results"]:
            if CN.search(x["webTitle"]):
                rows.append({"timestamp_utc": x["webPublicationDate"], "headline": x["webTitle"], "source": "The Guardian"})
        print("Guardian page", page, "of", resp["pages"], len(rows))
        if page >= resp["pages"]:
            break
        page += 1; time.sleep(0.5)
    df = pd.DataFrame(rows, columns=COLS); df.to_csv(f, index=False); return df

def google_week(query, a, b):
    q = f"{query} after:{a.strftime('%Y-%m-%d')} before:{b.strftime('%Y-%m-%d')}"
    url = f"https://news.google.com/rss/search?q={quote_plus(q)}&hl=en-US&gl=US&ceid=US:en"
    for attempt in range(4):
        try:
            r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            if r.status_code == 200 and "<rss" in r.text[:500]:
                rows = []
                for it in ET.fromstring(r.content).iter("item"):
                    title = it.findtext("title") or ""
                    src = it.findtext("source") or ""
                    if src and title.endswith(" - " + src):
                        title = title[: -len(" - " + src)]
                    try:
                        t = parsedate_to_datetime(it.findtext("pubDate"))
                    except Exception:
                        continue
                    rows.append({"timestamp_utc": pd.Timestamp(t).tz_convert("UTC"), "headline": title, "source": src})
                return rows
            print(f"    {a.date()}: HTTP {r.status_code}, retry {attempt + 1}")
        except Exception as e:
            print(f"    {a.date()}: {str(e)[:80]}, retry {attempt + 1}")
        time.sleep(5 * (attempt + 1))
    return None

def download_google():
    os.makedirs("raw/news_google", exist_ok=True)
    weeks = pd.date_range(pd.Timestamp(START) - pd.Timedelta(days=7), pd.Timestamp(END) + pd.Timedelta(days=7), freq="7D")
    pairs = list(zip(weeks[:-1], weeks[1:]))
    for k, (a, b) in enumerate(pairs, 1):
        f = f"raw/news_google/{a.date()}.csv"
        if os.path.exists(f):
            continue
        rows, ok = [], True
        for q in GOOGLE_QUERIES:
            res = google_week(q, a, b)
            if res is None:
                ok = False
            else:
                rows += res
            time.sleep(1)
        if ok:
            pd.DataFrame(rows, columns=COLS).to_csv(f, index=False)
        if k % 10 == 0 or k == len(pairs):
            print(f"  {k}/{len(pairs)} weeks ({a.date()}: {len(rows)} headlines)")
    frames = [pd.read_csv(f) for f in sorted(glob.glob("raw/news_google/*.csv"))]
    frames = [x for x in frames if not x.empty]
    if not frames:
        raise RuntimeError("Google News returned nothing. Try again later or set USE_GOOGLE = False.")
    out = pd.concat(frames, ignore_index=True)
    out["timestamp_utc"] = pd.to_datetime(out["timestamp_utc"], utc=True, format="mixed")
    lo = pd.Timestamp(START, tz="UTC") - pd.Timedelta(days=8)
    hi = pd.Timestamp(END, tz="UTC") + pd.Timedelta(days=1)
    return out[(out["timestamp_utc"] >= lo) & (out["timestamp_utc"] <= hi)]

## Collect headlines (NYT, Guardian, Google News). About 15 to 20 minutes

In [ ]:
hs = sheets.get("headlines")
if hs is not None and not hs.dropna(how="all").empty:
    tz = headline_tz()
    t = pd.to_datetime(hs.iloc[:, 0], errors="coerce")
    heads = pd.DataFrame({"timestamp_utc": t, "headline": hs.iloc[:, 1],
                          "source": hs.iloc[:, 2] if hs.shape[1] > 2 else "LSEG"}).dropna(subset=["timestamp_utc"])
    heads["timestamp_utc"] = heads["timestamp_utc"].dt.tz_localize(tz, ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC")
    heads = heads.dropna(subset=["timestamp_utc"])
    SRC_NOTE = "LSEG template headlines"
    print(f"Headlines from template (time zone {tz})")
else:
    parts = [nyt_archive(NYT_KEY), guardian(GUARDIAN_KEY)]
    SRC_NOTE = "New York Times Archive API and Guardian Open Platform (exact timestamps)"
    if USE_GOOGLE:
        g = download_google()
        g["timestamp_utc"] = pd.to_datetime(g["timestamp_utc"], utc=True, format="mixed")
        # Google gives the date only: treat as published 12:00 UTC (20:00 Shanghai), after the close
        g["timestamp_utc"] = g["timestamp_utc"].dt.normalize() + pd.Timedelta(hours=12)
        parts.append(g)
        SRC_NOTE += " plus Google News (date only, set to 20:00 Shanghai)"
    heads = pd.concat(parts, ignore_index=True)
    heads["timestamp_utc"] = pd.to_datetime(heads["timestamp_utc"], utc=True, format="mixed")
    heads = heads[~heads["headline"].astype(str).str.contains(EXCL.pattern, case=False, regex=True, na=False)]

n_raw = len(heads)
heads = clean_headlines(heads)
print(f"{n_raw} headlines, {len(heads)} after cleaning and de-duplication")
print(heads["source"].value_counts().head(8).to_string())
heads.sample(min(8, len(heads)), random_state=1)[["timestamp_utc", "source", "headline"]]

## Upload last session's score cache
Set `HAVE_CACHE = False` if you have no `scores_cache.csv`.

In [ ]:
HAVE_CACHE = True      # False if you have no scores_cache.csv from the earlier run
if HAVE_CACHE:
    from google.colab import files
    up = files.upload()
    open(CACHE, "wb").write(up[list(up)[0]])
    print("Cache loaded:", len(pd.read_csv(CACHE)), "rows")

## Scoring functions

In [ ]:
LABEL_LINE = re.compile(r"^\s*\[?(\d+)\]?\s*[:.)\-]\s*\**\s*(UP|DOWN|UNCERTAIN)\b", re.I | re.M)

# Ma, Lyu and Li (2024) instruction, applied to each numbered headline separately
BATCH_PROMPT = (PROMPT.split("\n\nHeadline:")[0] +
    "\n\nApply this instruction to each numbered headline below separately. Judge every "
    "headline on its own, using only that headline's information, independently of the others. "
    "Answer with exactly one line per headline in the format <number>: <UP, DOWN or UNCERTAIN>, "
    "and nothing else.\n\n{numbered}")

_last = [0.0]
def throttle():
    wait = 60.0 / REQUESTS_PER_MINUTE - (time.time() - _last[0])
    if wait > 0:
        time.sleep(wait)
    _last[0] = time.time()

from google import genai
from google.genai import types
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

GEN_CFG = [None, False]      # [working config, found yet]

def candidate_configs():
    base = dict(temperature=0, max_output_tokens=4096)
    yield types.GenerateContentConfig(**base)
    yield types.GenerateContentConfig(temperature=0)
    yield types.GenerateContentConfig(**base, thinking_config=types.ThinkingConfig(thinking_budget=0))
    yield None

def call_gemini(prompt):
    if GEN_CFG[1]:
        return client.models.generate_content(model=MODEL, contents=prompt, config=GEN_CFG[0]).text or ""
    last = None
    for cfg in candidate_configs():
        try:
            r = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
            GEN_CFG[0], GEN_CFG[1] = cfg, True
            return r.text or ""
        except Exception as e:
            last = e
            if "INVALID_ARGUMENT" not in str(e):
                raise
    raise last

def gemini_check():
    """One small test request before scoring. Stops with a clear message if key or model is wrong."""
    try:
        out = call_gemini("Reply with the single word OK.")
        print(f"Gemini check passed ({MODEL}): {out.strip()[:20]}")
    except Exception as e:
        print("Gemini check FAILED:", str(e)[:300])
        try:
            names = [m.name.replace("models/", "") for m in client.models.list()]
            print("Models your key can use:", [n for n in names if "flash" in n or "pro" in n][:20])
            print("Put one of these in MODEL (Settings cell) and run again.")
        except Exception as e2:
            print("Could not list models either, so the API key is probably the problem:", str(e2)[:200])
            print("Create a key at aistudio.google.com > Get API key and put it in Colab Secrets.")
        raise SystemExit("Stopped before scoring.")

def score_batch(batch):
    numbered = "\n".join(f"{i}. {h}" for i, h in enumerate(batch, 1))
    for attempt in range(6):
        throttle()
        try:
            text = call_gemini(BATCH_PROMPT.format(numbered=numbered))
            got = {int(n): lab.upper() for n, lab in LABEL_LINE.findall(text)}
            return {batch[i - 1]: got.get(i) for i in range(1, len(batch) + 1)}
        except Exception as e:
            msg = str(e)
            if "PerDay" in msg or "per day" in msg.lower():
                raise RuntimeError("DAILY_LIMIT")
            if "INVALID_ARGUMENT" in msg or "NOT_FOUND" in msg or "PERMISSION_DENIED" in msg:
                raise SystemExit("Gemini refused the request: " + msg[:300])
            wait = 10 * 2 ** attempt
            print(f"  retry in {wait}s: {msg[:150]}")
            time.sleep(wait)
    return {h: None for h in batch}

def headlines_to_score():
    h = heads.copy()
    h["week_idx"] = week_index(h["timestamp_utc"], wk)
    h = h[h["week_idx"] < len(wk)]
    if MAX_HEADLINES_PER_WEEK:
        src = h["source"].astype(str).str.lower()
        h["_pref"] = src.apply(lambda x: any(p in x for p in PREFERRED_SOURCES))
        h = h.sample(frac=1, random_state=42).sort_values("_pref", ascending=False, kind="stable")
        h = h[h.groupby("week_idx").cumcount() < MAX_HEADLINES_PER_WEEK]
        print(f"Capped at {MAX_HEADLINES_PER_WEEK} per week: {int(h['_pref'].sum())} of {len(h)} from preferred outlets")
        h = h.drop(columns="_pref")
    return h.sort_values("timestamp_utc")

def load_cache():
    if not os.path.exists(CACHE):
        return pd.DataFrame(columns=["headline", "label", "model"])
    c = pd.read_csv(CACHE)
    return c[c["model"] == MODEL].dropna(subset=["label"]).drop_duplicates("headline", keep="last")

def score_all(max_rounds=3):
    for rnd in range(1, max_rounds + 1):
        todo = sorted(set(to_score["headline"]) - set(load_cache()["headline"]))
        if not todo:
            break
        n_req = -(-len(todo) // BATCH_SIZE)
        print(f"Round {rnd}: {len(todo)} headlines, {n_req} requests, about {n_req / REQUESTS_PER_MINUTE:.0f} min")
        for b in range(n_req):
            batch = todo[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
            try:
                res = score_batch(batch)
            except RuntimeError:
                print("Daily request limit reached. Download scores_cache.csv and run again tomorrow.")
                return
            pd.DataFrame({"headline": list(res), "label": list(res.values()), "model": MODEL}).to_csv(
                CACHE, mode="a", header=not os.path.exists(CACHE), index=False)
            if (b + 1) % 10 == 0 or b + 1 == n_req:
                print(f"  {b + 1}/{n_req} requests done")
    c = load_cache()
    missing = len(set(to_score["headline"]) - set(c["headline"]))
    print("Label counts:", c["label"].value_counts().to_dict(), f"| unscored: {missing}")

## Score headlines with Gemini. About 15 to 20 minutes

In [ ]:
gemini_check()
to_score = headlines_to_score()
print(f"{len(to_score)} headlines selected for scoring")
score_all()

## Download the score cache

In [ ]:
# Download the scores so you can re-upload them in a later session
try:
    from google.colab import files; files.download(CACHE)
except ImportError:
    pass

## Weekly dataset and regressions

In [ ]:
import statsmodels.api as sm

def weekly_sentiment(wk):
    cache = load_cache()
    df = to_score.merge(cache[["headline", "label"]], on="headline", how="left")
    df["score"] = df["label"].map(SCORE_MAP)
    ts = pd.to_datetime(df["timestamp_utc"], utc=True)
    df["time_shanghai"] = ts.dt.tz_convert(TZ).dt.tz_localize(None)
    df = df[df["week_idx"] < len(wk)].copy()
    df["week_ending"] = wk["last_trading_day"].values[df["week_idx"].values]
    g = df.dropna(subset=["score"]).groupby("week_idx")["score"]
    agg = pd.DataFrame({"n_headlines": g.size(), "n_up": g.apply(lambda s: (s == 1).sum()),
                        "n_down": g.apply(lambda s: (s == -1).sum()),
                        "n_uncertain": g.apply(lambda s: (s == 0).sum()), "score_mean": g.mean()})
    agg["score_net"] = (agg["n_up"] - agg["n_down"]) / (agg["n_up"] + agg["n_down"]).replace(0, np.nan)
    out = wk.join(agg, how="left")
    out["n_headlines"] = out["n_headlines"].fillna(0).astype(int)
    return out, df

def run_regressions(d):
    specs = [("1 Contemporaneous", "erp", ["score_mean"]),
             ("2 Predictive", "erp_next", ["score_mean"]),
             ("3 Baseline (no sentiment)", "erp_next", ["erp"]),
             ("4 Baseline + sentiment", "erp_next", ["erp", "score_mean"])]
    rows = []
    for name, y, xs in specs:
        s = d[[y] + xs].dropna()
        m = sm.OLS(s[y], sm.add_constant(s[xs])).fit()
        for v in ["const"] + xs:
            rows.append({"model": name, "dependent": y, "variable": v, "coef": m.params[v],
                         "std_err": m.bse[v], "t_stat": m.tvalues[v], "p_value": m.pvalues[v],
                         "R2": m.rsquared, "adj_R2": m.rsquared_adj, "N": int(m.nobs)})
    return pd.DataFrame(rows)

d, scored = weekly_sentiment(wk)
d["erp_next"] = d["erp"].shift(-1)
d = d[(d["last_trading_day"] >= pd.Timestamp(START)) & (d["last_trading_day"] <= pd.Timestamp(END))].copy()
mkt = d.set_index("last_trading_day")[["erp", "erp_next"]]        # all sample weeks, kept for the robustness table
n_empty = int((d["n_headlines"] == 0).sum())
d = d[d["n_headlines"] > 0]
cols = ["last_trading_day", "close", "ret_log", "rf_pct_start", "rf_weekly", "erp", "erp_next",
        "n_headlines", "n_up", "n_down", "n_uncertain", "score_mean", "score_net"]
d = d[cols].rename(columns={"last_trading_day": "week_ending"})
reg = run_regressions(d)
print(f"{len(d)} weeks used, {n_empty} weeks without scored headlines dropped")
print(f"Weeks with under 10 headlines: {int((d['n_headlines'] < 10).sum())} | headlines per week: mean {d['n_headlines'].mean():.1f}, min {d['n_headlines'].min()}, max {d['n_headlines'].max()}")
if len(d) < 50:
    print("WARNING: fewer than 50 weeks. Extend the sample.")

def hit_rate(score, ret):
    m = score.notna() & ret.notna() & (score != 0)
    return float((np.sign(score[m]) == np.sign(ret[m])).mean()), int(m.sum())

hit_now, n_now = hit_rate(d["score_mean"], d["erp"])
hit_next, n_next = hit_rate(d["score_mean"], d["erp_next"])
corr_now = d["score_mean"].corr(d["erp"])
corr_next = d["score_mean"].corr(d["erp_next"])
extra = pd.DataFrame({
    "measure": ["Hit rate, same week", "Hit rate, next week", "Correlation, same week", "Correlation, next week"],
    "value": [hit_now, hit_next, corr_now, corr_next],
    "N": [n_now, n_next, d[["score_mean", "erp"]].dropna().shape[0], d[["score_mean", "erp_next"]].dropna().shape[0]]})
print(extra.round(3).to_string(index=False))
print("Hit rate = share of weeks where the sign of the score matches the sign of the ERP (50% = coin flip).")
reg[reg["variable"] != "const"].round(4)

## Robustness

In [ ]:
STOCK = re.compile(r"stock|share|equit|index|indices|rally|sell-?off|bear market|bull market|csi 300|shanghai composite|a-share|hang seng", re.I)

def robust_row(label, sub, min_headlines=0):
    g = sub.dropna(subset=["score"]).groupby("week_ending")["score"].agg(score_mean="mean", n="size")
    x = mkt.join(g, how="inner")
    x = x[x["n"] >= max(1, min_headlines)]
    out = {"version": label, "headlines": int(len(sub)), "weeks": len(x)}
    for key, y in [("predictive", "erp_next"), ("same_week", "erp")]:
        s = x[[y, "score_mean"]].dropna()
        m = sm.OLS(s[y], sm.add_constant(s["score_mean"])).fit()
        out[f"{key}_coef"] = m.params["score_mean"]
        out[f"{key}_t"] = m.tvalues["score_mean"]
    return out

sc = scored.dropna(subset=["score"])
is_cn = sc["headline"].str.contains(CN.pattern, case=False, regex=True)
is_stock = is_cn & sc["headline"].str.contains(STOCK.pattern, case=False, regex=True)
is_exact = sc["source"].astype(str).str.contains("new york times|guardian", case=False, regex=True)
rows_rb = [robust_row("All weeks", sc),
           robust_row("Weeks with at least 10 headlines", sc, 10),
           robust_row("China-related headlines only", sc[is_cn]),
           robust_row("China stock-market headlines only", sc[is_stock])]
if is_exact.any() and (~is_exact).any():
    rows_rb.append(robust_row("NYT and Guardian headlines only (exact timestamps)", sc[is_exact]))
    rows_rb.append(robust_row("Google News headlines only (market queries)", sc[~is_exact]))
rb = pd.DataFrame(rows_rb)
print("China-related: headline names China, Chinese, Beijing, Shanghai, yuan, PBOC, Hong Kong, CSI 300, Xi Jinping, Shenzhen, A-share or Hang Seng.")
rb.round(4)

## Chart

In [ ]:
import matplotlib.pyplot as plt
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(d["week_ending"], d["erp"] * 100, width=5, color="#9bb7d4", label="Weekly ERP (%)")
ax1.set_ylabel("ERP (%)")
ax2 = ax1.twinx()
ax2.plot(d["week_ending"], d["score_mean"], color="#c0392b", lw=1.8, label="Sentiment score")
ax2.set_ylabel("Mean sentiment score")
ax1.set_title(f"{INDEX}: weekly equity risk premium and LLM headline sentiment")
fig.legend(loc="upper left", bbox_to_anchor=(0.07, 0.88))
fig.tight_layout()
CHART = f"output/sentiment_vs_erp_{TAG}.png"
fig.savefig(CHART, dpi=200)
plt.show()

## Results summary

In [ ]:
def results_summary():
    lab = scored["label"].value_counts(normalize=True)
    lines = [
        f"Dependent variable: weekly equity risk premium of the {INDEX} (log return minus risk-free rate / 52; risk-free = {RF_LABEL}).",
        f"Sample: {d['week_ending'].min().date()} to {d['week_ending'].max().date()}, {len(d)} weeks, "
        f"{int(scored['score'].notna().sum())} headlines scored ({d['n_headlines'].mean():.1f} per week on average).",
        f"Headline source: {SRC_NOTE}.",
        f"LLM: Gemini {MODEL}, zero-shot, temperature 0, Ma, Lyu and Li (2024) prompt, {BATCH_SIZE} headlines per request, each judged separately.",
        f"Label shares: UP {lab.get('UP', 0):.1%}, DOWN {lab.get('DOWN', 0):.1%}, UNCERTAIN {lab.get('UNCERTAIN', 0):.1%}.",
        f"Weekly score_mean: mean {d['score_mean'].mean():.3f}, standard deviation {d['score_mean'].std():.3f}.",
        f"Weekly ERP: mean {d['erp'].mean():.4f}, standard deviation {d['erp'].std():.4f}.",
        "", "OLS regressions (coefficient, t-stat, p-value, adjusted R2, N):"]
    for name, g in reg.groupby("model", sort=False):
        for _, r in g[g["variable"] != "const"].iterrows():
            lines.append(f"  {name}: {r['dependent']} on {r['variable']}: coef {r['coef']:.4f}, t {r['t_stat']:.2f}, "
                         f"p {r['p_value']:.3f}, adj R2 {r['adj_R2']:.4f}, N {r['N']}")
    lines += ["", f"Hit rate same week {hit_now:.1%} (N {n_now}), next week {hit_next:.1%} (N {n_next}). 50% = coin flip.",
              f"Correlation of score with ERP: same week {corr_now:.3f}, next week {corr_next:.3f}.", "",
              "Robustness (version, weeks, predictive t, same-week t):"]
    for _, r in rb.iterrows():
        lines.append(f"  {r['version']}: {int(r['weeks'])} weeks, predictive t {r['predictive_t']:.2f}, same-week t {r['same_week_t']:.2f}")
    lines += ["", "Most positive sentiment weeks (week, score, ERP that week, ERP next week):"]
    for _, r in d.nlargest(3, "score_mean").iterrows():
        lines.append(f"  {r['week_ending'].date()}: {r['score_mean']:.2f}, {r['erp']:.4f}, {r['erp_next']:.4f}")
    lines.append("Most negative sentiment weeks:")
    for _, r in d.nsmallest(3, "score_mean").iterrows():
        lines.append(f"  {r['week_ending'].date()}: {r['score_mean']:.2f}, {r['erp']:.4f}, {r['erp_next']:.4f}")
    lines += ["", "Known design facts: headline-only information; the LLM may have seen news from the sample period "
              "in training (look-ahead risk); plain OLS standard errors; headlines are scored in batches of "
              f"{BATCH_SIZE}; the course brief says this is not an econometrics assignment and asks for univariate regressions."]
    return "\n".join(lines)

SUMMARY = results_summary()
print(SUMMARY)

## Debate (switched off)

In [ ]:
%%script false --no-raise-error
STYLE = ("Write in plain English prose with short sentences and British spelling. No bullet points, no headings, "
         "no bold. Use only the numbers in the results summary. Do not invent data, papers or figures. "
         "Write numbers as digits, exactly as they appear in the summary. Maximum 220 words.")

PRO = ("You are a quantitative researcher. Argue that zero-shot LLM headline sentiment is useful for explaining "
       "and forecasting the weekly Shanghai Composite equity risk premium in this study. Build the strongest "
       "honest case from the results. Where results are weak, explain why the approach still has value. " + STYLE)
CON = ("You are a sceptical econometrician. Argue that this study does not show that LLM headline sentiment is "
       "useful for forecasting the weekly Shanghai Composite equity risk premium. Attack statistical significance, "
       "economic size, sample length, look-ahead bias, headline noise, batching of headlines and the features of "
       "the Chinese market, as far as the results support it. Be fair and specific. " + STYLE)
JUDGE = ("You are a neutral finance professor judging a debate about a study. Weigh both sides strictly "
         "against the results summary, not against rhetoric. Write plain prose, short sentences, British spelling. "
         "Give, in this order: 1) Verdict: one of SUPPORTED, PARTLY SUPPORTED, NOT SUPPORTED, with two sentences "
         "of reasoning. 2) Score out of 10 for each side. 3) The two strongest points of each side. 4) What the "
         "author should say in its conclusion about implications for financial forecasting, in three "
         "sentences. 5) Three concrete limitations to present. Maximum 400 words.")

def agent(role_prompt, task, temperature=0.3):
    for attempt in range(5):
        throttle()
        try:
            cfg = types.GenerateContentConfig(temperature=temperature, max_output_tokens=2048)
            try:
                r = client.models.generate_content(
                    model=MODEL, contents=f"{role_prompt}\n\nRESULTS SUMMARY\n{SUMMARY}\n\nTASK\n{task}", config=cfg)
            except Exception as e:
                if "INVALID_ARGUMENT" not in str(e):
                    raise
                r = client.models.generate_content(
                    model=MODEL, contents=f"{role_prompt}\n\nRESULTS SUMMARY\n{SUMMARY}\n\nTASK\n{task}", config=GEN_CFG[0])
            if r.text:
                return r.text.strip()
        except Exception as e:
            print(f"  retry: {str(e)[:120]}")
            time.sleep(10 * 2 ** attempt)
    return "(no answer)"

debate = []
def say(who, text):
    debate.append((who, text))
    print(f"\n===== {who} =====\n{text}")

pro1 = agent(PRO, "Give your opening argument.")
say("PRO, opening", pro1)
con1 = agent(CON, "Give your opening argument.")
say("CON, opening", con1)
pro2 = agent(PRO, f"Your opponent argued:\n{con1}\n\nRebut their strongest points. Concede what is true.")
say("PRO, rebuttal", pro2)
con2 = agent(CON, f"Your opponent argued:\n{pro1}\n\nRebut their strongest points. Concede what is true.")
say("CON, rebuttal", con2)
transcript = "\n\n".join(f"{w}:\n{t}" for w, t in debate)
verdict = agent(JUDGE, f"DEBATE TRANSCRIPT\n{transcript}\n\nGive your judgement.", temperature=0)
say("JUDGE, verdict", verdict)

DEBATE_MD = f"output/debate_{TAG}.md"
with open(DEBATE_MD, "w") as f:
    f.write("# Debate on the results\n\n## Results summary\n\n```\n" + SUMMARY + "\n```\n\n")
    for who, text in debate:
        f.write(f"## {who}\n\n{text}\n\n")
print("\nSaved", DEBATE_MD)

## Excel workbook

In [ ]:
readme = pd.DataFrame({"item": ["Index", "Sample", "Weeks used", "Weeks dropped", "Headlines scored", "Headline source",
                                "Timestamps", "LLM", "Prompt", "erp", "erp_next", "score_mean", "score_net", "Week timing"],
                       "value": [INDEX, f"{d['week_ending'].min().date()} to {d['week_ending'].max().date()}",
                                 len(d), n_empty, int(scored["score"].notna().sum()), SRC_NOTE,
                                 "NYT and Guardian: exact publication time. Google News (if used): date only, set to 20:00 Shanghai.",
                                 f"Gemini / {MODEL}, {BATCH_SIZE} headlines per request", BATCH_PROMPT,
                                 f"ln(P_t / P_t-1) minus risk-free rate ({RF_LABEL}) at start of week / 52",
                                 "erp of the following week",
                                 "average headline score: UP=+1, UNCERTAIN=0, DOWN=-1",
                                 "(UP - DOWN) / (UP + DOWN)",
                                 "headlines up to 15:00 Shanghai on the week's last trading day"]})
heads_out = scored[["week_ending", "time_shanghai", "source", "headline", "label", "score"]].sort_values("time_shanghai")

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as xw:
    readme.to_excel(xw, sheet_name="readme", index=False)
    extra.to_excel(xw, sheet_name="hit_rate_corr", index=False)
    if "debate" in globals() and debate:
        pd.DataFrame(debate, columns=["speaker", "text"]).to_excel(xw, sheet_name="debate", index=False)
    d.to_excel(xw, sheet_name="weekly", index=False)
    reg.to_excel(xw, sheet_name="regressions", index=False)
    rb.to_excel(xw, sheet_name="robustness", index=False)
    heads_out.to_excel(xw, sheet_name="headlines", index=False)
    for ws in xw.book.worksheets:
        for col in ws.columns:
            w_ = max(len(str(c.value or "")) for c in col[:200])
            ws.column_dimensions[col[0].column_letter].width = min(max(10, w_ + 2), 70)
print("Saved", OUTPUT_XLSX)
try:
    from google.colab import files
    files.download(OUTPUT_XLSX); files.download(CHART)
    if "DEBATE_MD" in globals() and os.path.exists(DEBATE_MD): files.download(DEBATE_MD)
except ImportError:
    pass

## Newey-West, PMI models and out-of-sample test

In [ ]:
# Run after the "Weekly dataset and regressions" cell and the robustness cell. Uses d, scored, reg, sm, np, pd.
from scipy import stats

dd = d.sort_values("week_ending").reset_index(drop=True).copy()
dd["d_rf"] = dd["rf_pct_start"].diff()          # change in the 1-year yield since last week (percentage points)
if pmi is not None:
    rel = pmi.copy(); rel.index = rel.index.to_period("M").to_timestamp("M")   # NBS releases on the last day of the month
    rel = rel[~rel.index.duplicated(keep="last")]
    dd["pmi"] = rel.reindex(pd.to_datetime(dd["week_ending"]), method="ffill").values

def fit(y, xs, hac=True):
    s = dd[[y] + xs].dropna()
    X = sm.add_constant(s[xs])
    m = sm.OLS(s[y], X).fit(cov_type="HAC", cov_kwds={"maxlags": 2}) if hac else sm.OLS(s[y], X).fit()
    return m, len(s)

specs = [("1 Same week", "erp", ["score_mean"]),
         ("2 Predictive", "erp_next", ["score_mean"]),
         ("3 Baseline", "erp_next", ["erp"]),
         ("4 Baseline + score", "erp_next", ["erp", "score_mean"]),
         ("5 Baseline + score + change in yield", "erp_next", ["erp", "score_mean", "d_rf"])]
if "pmi" in dd:
    specs += [("6 PMI baseline", "erp_next", ["pmi"]),
              ("7 PMI baseline + score", "erp_next", ["pmi", "score_mean"])]
rows = []
for name, y, xs in specs:
    m, n = fit(y, xs)
    for v in xs:
        rows.append({"model": name, "dependent": y, "variable": v, "coef": m.params[v], "std_err_HAC": m.bse[v],
                     "t_stat_HAC": m.tvalues[v], "p_value_HAC": m.pvalues[v], "adj_R2": m.rsquared_adj, "N": n})
reg_hac = pd.DataFrame(rows)
print("Newey-West (2 lags) standard errors, plus model 5 with the change in the risk-free rate")
print(reg_hac.round(4).to_string(index=False))

# ---- Out-of-sample: expanding window, first forecast after half the sample ----
FIRST = max(30, len(dd) // 2)
def oos(spec_cols):
    preds, actual, dates = [], [], []
    for t in range(FIRST, len(dd) - 1):
        train = dd.iloc[:t][["erp_next"] + spec_cols].dropna() if spec_cols else dd.iloc[:t][["erp_next"]].dropna()
        x_t = dd.iloc[t][spec_cols] if spec_cols else None
        if spec_cols and x_t.isna().any():
            continue
        if spec_cols:
            m = sm.OLS(train["erp_next"], sm.add_constant(train[spec_cols])).fit()
            p = float(m.params["const"] + (m.params[spec_cols] * x_t).sum())
        else:
            p = float(train["erp_next"].mean())
        preds.append(p); actual.append(dd.iloc[t]["erp_next"]); dates.append(dd.iloc[t]["week_ending"])
    return pd.Series(preds, index=dates), pd.Series(actual, index=dates)

models = {"Historical mean": [], "Baseline (last week's ERP)": ["erp"],
          "Baseline + score": ["erp", "score_mean"], "Baseline + score + change in yield": ["erp", "score_mean", "d_rf"],
          "Score only": ["score_mean"]}
if "pmi" in dd:
    models.update({"PMI baseline": ["pmi"], "PMI baseline + score": ["pmi", "score_mean"]})
P, Y = {}, {}
for k, cols in models.items():
    P[k], Y[k] = oos(cols)
common = sorted(set.intersection(*[set(v.index) for v in P.values()]))
y_true = Y["Historical mean"].loc[common]
sse_mean = float(((y_true - P["Historical mean"].loc[common]) ** 2).sum())
out_rows = []
for k in models:
    e = y_true - P[k].loc[common]
    sign_ok = (np.sign(P[k].loc[common]) == np.sign(y_true)).mean()
    out_rows.append({"model": k, "forecasts": len(common), "RMSE_pct_points": 100 * float(np.sqrt((e ** 2).mean())),
                     "OOS_R2_vs_mean": 1 - float((e ** 2).sum()) / sse_mean, "hit_rate_sign": float(sign_ok)})
oos_tbl = pd.DataFrame(out_rows)
print(f"\nOut-of-sample forecasts of next week's ERP, expanding window, {len(common)} forecasts, first after {FIRST} weeks")
print(oos_tbl.round(4).to_string(index=False))

# Clark-West test: does adding the score beat the nested baseline?
def clark_west(small, large):
    e1 = y_true - P[small].loc[common]; e2 = y_true - P[large].loc[common]
    f = e1 ** 2 - (e2 ** 2 - (P[small].loc[common] - P[large].loc[common]) ** 2)
    t = f.mean() / (f.std(ddof=1) / np.sqrt(len(f)))
    return float(t), float(1 - stats.norm.cdf(t))
cw_t, cw_p = clark_west("Baseline (last week's ERP)", "Baseline + score")
print(f"Clark-West, baseline vs baseline + score: t {cw_t:.2f}, one-sided p {cw_p:.3f}")
if "pmi" in dd:
    cw_t2, cw_p2 = clark_west("PMI baseline", "PMI baseline + score")
    print(f"Clark-West, PMI baseline vs PMI baseline + score: t {cw_t2:.2f}, one-sided p {cw_p2:.3f}")
print("OOS R2 below 0 means the model forecasts worse than the historical mean.")

## Blind hand check sample

In [ ]:
# Fill the my_label column by hand in the Excel sheet (UP, DOWN or UNCERTAIN). The agreement rate is a formula.
label_key = scored.dropna(subset=["label"]).sample(50, random_state=7)[["source", "headline", "label"]].reset_index(drop=True)
label_key = label_key.rename(columns={"label": "gemini_label"})
check = label_key[["source", "headline"]].copy()          # what the raters see: no Gemini label
check["rater_1"] = ""; check["rater_2"] = ""
print(check.head(5).to_string(index=False))

## Add sheets to the workbook

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
    reg_hac.to_excel(xw, sheet_name="regressions_hac", index=False)
    oos_tbl.to_excel(xw, sheet_name="out_of_sample", index=False)
    check.to_excel(xw, sheet_name="label_check", index=False)
    label_key.to_excel(xw, sheet_name="label_key", index=False)
    lin = dd[["erp", "score_mean", "erp_next"]].dropna().reset_index(drop=True)
    lin.to_excel(xw, sheet_name="linest", index=False, startrow=14)

wb = load_workbook(OUTPUT_XLSX)
ws = wb["linest"]
n = len(lin); r0, r1 = 16, 15 + n
lin1 = dd[["erp", "score_mean"]].dropna().reset_index(drop=True)      # 107-week block for model 1, written in columns L:M
ws["L15"], ws["M15"] = f"erp ({len(lin1)} weeks)", f"score_mean ({len(lin1)} weeks)"
for i_, (e_, s_) in enumerate(lin1.itertuples(index=False)):
    ws.cell(row=16 + i_, column=12, value=float(e_)); ws.cell(row=16 + i_, column=13, value=float(s_))
n1 = len(lin1); Y1, X1 = f"L16:L{15 + n1}", f"M16:M{15 + n1}"                     # data rows (header on row 15)
A, B, C = f"A{r0}:A{r1}", f"B{r0}:B{r1}", f"C{r0}:C{r1}"
ws["A1"] = "LINEST regressions (formulas). Data from row 15: erp (A), score_mean (B), erp_next (C) for models 2 to 4, and a 107-week block in L:M for model 1."
ws["A1"].font = Font(bold=True)
hdr = ["Model", "Coef score", "Coef erp", "Intercept", "SE score", "SE erp", "t score", "p score", "R2", "N"]
for j, h in enumerate(hdr, 1):
    ws.cell(row=3, column=j, value=h).font = Font(bold=True)
def L(y, x, r, c): return f"INDEX(LINEST({y},{x},TRUE,TRUE),{r},{c})"
# model 1: erp ~ score ; model 2: erp_next ~ score ; model 3: erp_next ~ erp ; model 4: erp_next ~ erp + score (x = A:B)
ws["A4"], ws["A5"], ws["A6"], ws["A7"] = "1 Same week (erp on score)", "2 Predictive (erp_next on score)", "3 Baseline (erp_next on erp)", "4 Baseline + score"
for row, (y, x) in zip([4, 5], [(Y1, X1), (C, B)]):
    ws[f"B{row}"] = "=" + L(y, x, 1, 1); ws[f"D{row}"] = "=" + L(y, x, 1, 2)
    ws[f"E{row}"] = "=" + L(y, x, 2, 1)
    ws[f"G{row}"] = f"=B{row}/E{row}"; ws[f"I{row}"] = "=" + L(y, x, 3, 1)
    ws[f"H{row}"] = f"=TDIST(ABS(G{row}),{L(y, x, 4, 2)},2)"
ws["C6"] = "=" + L(C, A, 1, 1); ws["D6"] = "=" + L(C, A, 1, 2); ws["F6"] = "=" + L(C, A, 2, 1)
ws["G6"] = "=C6/F6"; ws["I6"] = "=" + L(C, A, 3, 1); ws["H6"] = f"=TDIST(ABS(G6),{L(C, A, 4, 2)},2)"
X2 = f"A{r0}:B{r1}"                                    # columns erp, score_mean: LINEST returns score first, then erp, then intercept
ws["B7"] = "=" + L(C, X2, 1, 1); ws["C7"] = "=" + L(C, X2, 1, 2); ws["D7"] = "=" + L(C, X2, 1, 3)
ws["E7"] = "=" + L(C, X2, 2, 1); ws["F7"] = "=" + L(C, X2, 2, 2)
ws["G7"] = "=B7/E7"; ws["I7"] = "=" + L(C, X2, 3, 1); ws["H7"] = f"=TDIST(ABS(G7),{L(C, X2, 4, 2)},2)"
ws["J4"] = f"=COUNT({Y1})"
for row in range(5, 8):
    ws[f"J{row}"] = f"=COUNT({C})"
ws["A9"] = f"Model 1 uses its own {len(lin1)}-week block in columns L:M, so N matches the regressions sheet. Models 2 to 4 use the {n} weeks that have a next-week ERP (columns A:C). Plain standard errors."
ws2 = wb["label_check"]
from openpyxl.worksheet.datavalidation import DataValidation
dv = DataValidation(type="list", formula1='"UP,DOWN,UNCERTAIN"', allow_blank=True)
ws2.add_data_validation(dv); dv.add("C2:D51")
ws2["F1"] = "Two people fill rater_1 and rater_2 separately. Do not open label_key until both are done."
wb["label_key"].sheet_state = "hidden"
wb.save(OUTPUT_XLSX)
print("Added regressions_hac, out_of_sample, label_check and linest sheets to", OUTPUT_XLSX)
try:
    from google.colab import files
    files.download(OUTPUT_XLSX)
except ImportError:
    pass

## Optional: bull, bear and judge scoring (switched off, needs 2 to 3 days of free-tier quota)

In [ ]:
%%script false --no-raise-error
# Multi-agent scoring: Bull, Bear and Judge prompts applied to every headline. Run after the scoring cell.
MA_SOURCE_TYPE = "news headline"
MA_MARKET = "the Chinese stock market (Shanghai Composite)"
MA_HORIZON = "week"
MA_BATCH = 20                      # headlines per request. 1 = one headline per request (exact prompts, about 5,900 requests)
MA_CACHE = "debate_scores_cache.csv"

BULL = ("Forget all previous instructions. You are an optimistic financial analyst.\n\n"
        "Below is a {source_type} published on {date}.\n\n"
        "Using ONLY the information in this text, give the 3 strongest arguments why {market} could GO UP over the next {horizon}.\n"
        "If the text contains no positive arguments, say so honestly.\n\n"
        "Answer as a numbered list of 3 short sentences. No introduction, no conclusion.\n\n"
        "Text:\n{text}")
BEAR = ("Forget all previous instructions. You are a pessimistic financial analyst.\n\n"
        "Below is a {source_type} published on {date}.\n\n"
        "Using ONLY the information in this text, give the 3 strongest arguments why {market} could GO DOWN over the next {horizon}.\n"
        "If the text contains no negative arguments, say so honestly.\n\n"
        "Answer as a numbered list of 3 short sentences. No introduction, no conclusion.\n\n"
        "Text:\n{text}")
JUDGE_P = ("Forget all previous instructions. You are a neutral and experienced financial expert acting as a judge.\n\n"
           "Below is a {source_type} published on {date}, followed by an optimistic and a pessimistic analysis of it.\n\n"
           "Weigh both sides, check them against the original text, and decide the overall sentiment for {market} over the next {horizon}.\n\n"
           "Choose one score:\n+2 = very positive\n+1 = positive\n 0 = neutral or balanced\n-1 = negative\n-2 = very negative\n\n"
           "Answer ONLY in this exact format:\nSCORE: <number>\nREASON: <one short sentence>\n\n"
           "Original text:\n{text}\n\nOptimistic analysis:\n{bull_answer}\n\nPessimistic analysis:\n{bear_answer}")

_fmt = dict(source_type=MA_SOURCE_TYPE, market=MA_MARKET, horizon=MA_HORIZON)
BATCH_RULE = ("\n\nThis request contains several numbered items. Apply the instruction above to each item separately, "
              "using only that item's information. Start every answer with a line containing only '## <number>' and give the "
              "answer for that item below it, in the format requested above. Do not use bold or other markdown. "
              "Write nothing else.\n\n")

# Header line of an answer: '## 3', '### 3', '**3**', '[3]', '3:' or 'Item 3' on a line of its own
HEADER = re.compile(r"(?m)^[ \t]*(?:#+[ \t]*)?\**[ \t]*(?:item |headline )?\[?(\d{1,3})\]?[ \t]*[:.)]?[ \t]*\**[ \t]*$", re.I)
SCORE_RE = re.compile(r"SCORE\**[ \t]*[:=]?\**[ \t]*\**([+\-−–]?[ \t]*\d)", re.I)
_shown = [0]

def ma_call(prompt):
    for attempt in range(6):
        throttle()
        try:
            return call_gemini(prompt)
        except Exception as e:
            msg = str(e)
            if "PerDay" in msg or "per day" in msg.lower():
                raise RuntimeError("DAILY_LIMIT")
            if "INVALID_ARGUMENT" in msg or "NOT_FOUND" in msg or "PERMISSION_DENIED" in msg:
                raise SystemExit("Gemini refused the request: " + msg[:300])
            wait = 10 * 2 ** attempt
            print(f"  retry in {wait}s: {msg[:150]}")
            time.sleep(wait)
    return ""

def split_blocks(text):
    parts = HEADER.split(text or "")
    return {int(parts[i]): parts[i + 1].strip() for i in range(1, len(parts) - 1, 2)}

def parse_judge(body):
    body = body or ""
    m = SCORE_RE.search(body)
    r = re.search(r"REASON\**[ \t]*[:=]?\**[ \t]*(.+)", body, re.I)
    if not m:
        return None, ""
    v = int(m.group(1).replace("−", "-").replace("–", "-").replace(" ", "").replace("\t", ""))
    return (v if -2 <= v <= 2 else None), (r.group(1).strip(" *") if r else "")

def show_once(tag, raw):
    if _shown[0] < 3:
        _shown[0] += 1
        print(f"  could not read the {tag} answer. First 400 characters:\n{(raw or '(empty)')[:400]}\n")

def ma_batch(items):
    """items: list of (headline, date). Returns {headline: (bull, bear, score, reason)} for the items that were read correctly."""
    if MA_BATCH == 1:
        h, dt = items[0]
        bull = ma_call(BULL.format(date=dt, text=h, **_fmt)).strip()
        bear = ma_call(BEAR.format(date=dt, text=h, **_fmt)).strip()
        out = ma_call(JUDGE_P.format(date=dt, text=h, bull_answer=bull, bear_answer=bear, **_fmt))
        sc, why = parse_judge(out)
        return {h: (bull, bear, sc, why)} if sc is not None else {}
    listing = "\n".join(f"[{i}] (published {dt}) {h}" for i, (h, dt) in enumerate(items, 1))
    head = lambda t: t.split("\n\nText:\n")[0].format(date="the date shown for it", **_fmt)
    raw_bull = ma_call(head(BULL) + BATCH_RULE + listing); bull = split_blocks(raw_bull)
    raw_bear = ma_call(head(BEAR) + BATCH_RULE + listing); bear = split_blocks(raw_bear)
    if len(bull) < len(items) // 2: show_once("bull", raw_bull)
    if len(bear) < len(items) // 2: show_once("bear", raw_bear)
    if not bull or not bear:                      # nothing usable, skip the judge call
        return {}
    jhead = JUDGE_P.split("\n\nOriginal text:\n")[0].format(date="the date shown for it", **_fmt)
    jlist = "\n\n".join(f"[{i}] (published {dt}) {h}\nOptimistic analysis: {bull.get(i, '(missing)')}\n"
                        f"Pessimistic analysis: {bear.get(i, '(missing)')}" for i, (h, dt) in enumerate(items, 1))
    raw_j = ma_call(jhead + BATCH_RULE + jlist); judged = split_blocks(raw_j)
    if not judged:                                # fall back to the order of the SCORE lines
        found = [m.group(1) for m in SCORE_RE.finditer(raw_j)]
        if len(found) == len(items):
            judged = {i: f"SCORE: {g}" for i, g in enumerate(found, 1)}
    if len(judged) < len(items) // 2: show_once("judge", raw_j)
    res = {}
    for i, (h, dt) in enumerate(items, 1):
        if i not in bull or i not in bear:
            continue                              # no bull or bear text, so leave it for the next round
        sc, why = parse_judge(judged.get(i))
        if sc is not None:
            res[h] = (bull[i], bear[i], sc, why)
    return res

def load_ma_cache():
    if not os.path.exists(MA_CACHE):
        return pd.DataFrame(columns=["headline", "bull", "bear", "score_ma", "reason", "model"])
    c = pd.read_csv(MA_CACHE)
    return c[c["model"] == MODEL].dropna(subset=["score_ma"]).drop_duplicates("headline", keep="last")

def score_ma_all(max_rounds=4):
    dates = to_score.assign(date=pd.to_datetime(to_score["timestamp_utc"], utc=True).dt.tz_convert(TZ).dt.strftime("%Y-%m-%d"))
    date_of = dates.drop_duplicates("headline").set_index("headline")["date"]
    for rnd in range(1, max_rounds + 1):
        todo = sorted(set(to_score["headline"]) - set(load_ma_cache()["headline"]))
        if not todo:
            break
        n_req = -(-len(todo) // MA_BATCH)
        print(f"Round {rnd}: {len(todo)} headlines, up to {n_req * 3} requests, about {n_req * 3 / REQUESTS_PER_MINUTE:.0f} min")
        for b in range(n_req):
            batch = [(h, date_of[h]) for h in todo[b * MA_BATCH:(b + 1) * MA_BATCH]]
            try:
                res = ma_batch(batch)
            except RuntimeError:
                print("Daily request limit reached. Download", MA_CACHE, "and run this cell again tomorrow.")
                return
            if res:
                pd.DataFrame([{"headline": h, "bull": v[0], "bear": v[1], "score_ma": v[2], "reason": v[3], "model": MODEL}
                              for h, v in res.items()]).to_csv(MA_CACHE, mode="a", header=not os.path.exists(MA_CACHE), index=False)
            if (b + 1) % 5 == 0 or b + 1 == n_req:
                print(f"  {b + 1}/{n_req} batches done, {len(load_ma_cache())} headlines scored so far")
    c = load_ma_cache()
    print("Judge scores:", c["score_ma"].value_counts().sort_index().to_dict(),
          f"| unscored: {len(set(to_score['headline']) - set(c['headline']))}")

# To continue an earlier run, upload debate_scores_cache.csv first:
# from google.colab import files; up = files.upload(); open(MA_CACHE, "wb").write(up[list(up)[0]])
score_ma_all()
try:
    from google.colab import files; files.download(MA_CACHE)
except ImportError:
    pass

## Optional: compare the two scoring methods (switched off)

In [ ]:
%%script false --no-raise-error
# Multi-agent score by week, regressions and comparison with the single-prompt score. Run after the cell above.
ma = load_ma_cache()[["headline", "score_ma"]]
if len(ma) < 0.95 * len(to_score):
    print(f"WARNING: only {len(ma)} of {len(to_score)} headlines have a judge score. Run the cell above again until it is complete.")
sc_ma = scored.merge(ma, on="headline", how="left")
g_ma = sc_ma.dropna(subset=["score_ma"]).groupby("week_ending")["score_ma"].mean()

d_ma = d.copy()
d_ma["score_mean"] = d_ma["week_ending"].map(g_ma)          # weekly mean judge score, -2 to +2
d_ma = d_ma.dropna(subset=["score_mean"])
reg_ma = run_regressions(d_ma)

def hac_t(df, y, xs):
    s = df[[y] + xs].dropna()
    m = sm.OLS(s[y], sm.add_constant(s[xs])).fit(cov_type="HAC", cov_kwds={"maxlags": 2})
    return m.tvalues["score_mean"]

def pick(r, model, col):
    return float(r[(r["model"] == model) & (r["variable"] == "score_mean")][col].iloc[0])

h1, n1 = hit_rate(d["score_mean"], d["erp"]);     h2, n2 = hit_rate(d["score_mean"], d["erp_next"])
m1, k1 = hit_rate(d_ma["score_mean"], d_ma["erp"]); m2, k2 = hit_rate(d_ma["score_mean"], d_ma["erp_next"])
both = sc_ma.dropna(subset=["score", "score_ma"])
comp = pd.DataFrame({
    "measure": ["Headlines scored", "Weeks used", "Same-week coefficient", "Same-week t", "Same-week t (Newey-West)",
                "Predictive coefficient", "Predictive t", "Predictive t (Newey-West)", "Adj. R2 predictive",
                "Hit rate, same week", "Hit rate, next week"],
    "single prompt (UP/DOWN/UNCERTAIN)": [int(scored["score"].notna().sum()), len(d), pick(reg, "1 Contemporaneous", "coef"),
        pick(reg, "1 Contemporaneous", "t_stat"), hac_t(d, "erp", ["score_mean"]), pick(reg, "2 Predictive", "coef"),
        pick(reg, "2 Predictive", "t_stat"), hac_t(d, "erp_next", ["score_mean"]), pick(reg, "2 Predictive", "adj_R2"), h1, h2],
    "bull / bear / judge (-2 to +2)": [int(sc_ma["score_ma"].notna().sum()), len(d_ma), pick(reg_ma, "1 Contemporaneous", "coef"),
        pick(reg_ma, "1 Contemporaneous", "t_stat"), hac_t(d_ma, "erp", ["score_mean"]), pick(reg_ma, "2 Predictive", "coef"),
        pick(reg_ma, "2 Predictive", "t_stat"), hac_t(d_ma, "erp_next", ["score_mean"]), pick(reg_ma, "2 Predictive", "adj_R2"), m1, m2]})
print(comp.round(4).to_string(index=False))
agree = float((np.sign(both["score"]) == np.sign(both["score_ma"])).mean())
print(f"\nHeadline level: correlation of the two scores {both['score'].corr(both['score_ma']):.2f}, same sign for {agree:.1%} of headlines")
print("Judge score shares:", (sc_ma["score_ma"].value_counts(normalize=True).sort_index().round(3)).to_dict())

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
    comp.to_excel(xw, sheet_name="multi_agent", index=False)
    reg_ma.to_excel(xw, sheet_name="multi_agent_reg", index=False)
    load_ma_cache()[["headline", "bull", "bear", "score_ma", "reason"]].to_excel(xw, sheet_name="multi_agent_headlines", index=False)
print("Added multi_agent, multi_agent_reg and multi_agent_headlines sheets to", OUTPUT_XLSX)

## Kappa for the blind hand check (switched off until both raters are done)

In [ ]:
%%script false --no-raise-error
# Run after two people have filled rater_1 and rater_2 in the label_check sheet, without opening label_key.
from sklearn.metrics import cohen_kappa_score
from google.colab import files
up = files.upload(); fn = list(up)[0]
lc = pd.read_excel(fn, sheet_name="label_check").merge(
     pd.read_excel(fn, sheet_name="label_key")[["headline", "gemini_label"]], on="headline")
for c in ["rater_1", "rater_2", "gemini_label"]:
    lc[c] = lc[c].astype(str).str.upper().str.strip()
for a, b in [("rater_1", "rater_2"), ("rater_1", "gemini_label"), ("rater_2", "gemini_label")]:
    print(f"{a} vs {b}: agreement {(lc[a] == lc[b]).mean():.1%}, kappa {cohen_kappa_score(lc[a], lc[b]):.2f}")
print(pd.crosstab(lc["rater_1"], lc["gemini_label"]))